# Exercise 4: Naive matrix multiplication: solution

Solution to Exercise 4 in `matrix_multiplication.ipynb`.

In [ ]:
using Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

In [ ]:
using KernelAbstractions

# Pick your hardware: :NVIDIA, :AMD, :Metal, or :CPU
DEV = :NVIDIA

if DEV == :NVIDIA
    using CUDA
    ArrayKA = CUDA.CuArray
    Backend = CUDA.CUDABackend()
elseif DEV == :AMD
    using AMDGPU
    ArrayKA = AMDGPU.ROCArray
    Backend = AMDGPU.ROCBackend()
elseif DEV == :Metal
    using Metal
    ArrayKA = Metal.MtlArray
    Backend = Metal.MetalBackend()
elseif DEV == :CPU
    ArrayKA = Array
    Backend = CPU()
else
    error("Unknown DEV = $DEV. Use :NVIDIA, :AMD, :Metal, or :CPU.")
end

In [ ]:
@kernel function MatrixMultiplication_kernel!(C, @Const(A), @Const(B))
    row, col = @index(Global, NTuple)

    acc = zero(eltype(C))
    for k in 1:size(A, 2)
        @inbounds acc += A[row, k] * B[k, col]
    end

    @inbounds C[row, col] = acc
end

In [ ]:
matrix_size = 1024
A = rand(Float32, matrix_size, matrix_size)
B = rand(Float32, matrix_size, matrix_size)

DA = allocate(Backend, Float32, matrix_size, matrix_size)
copyto!(DA, A)
DB = allocate(Backend, Float32, matrix_size, matrix_size)
copyto!(DB, B)
DC = KernelAbstractions.zeros(Backend, Float32, matrix_size, matrix_size)

kernel! = MatrixMultiplication_kernel!(Backend, (32, 32))
kernel!(DC, DA, DB, ndrange=size(DC))
KernelAbstractions.synchronize(Backend)

isapprox(Array(DC), A * B)